In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# 1. get the current directory (which is '.../notebooks')
cwd = Path.cwd() 

# 2. set ROOT to the parent folder ('.../Final_project csci 161')
ROOT = cwd.parent 

# 3. define PROC and FIG based on this correct ROOT
PROC = ROOT / "data" / "processed"
FIG  = ROOT / "figures"
RAW  = ROOT / "data" / "raw"


print(f"ROOT: {ROOT}")
print(f"PROC: {PROC}")
print(f"FIG:  {FIG}")

# for double checking. This will now print 'True'
print(f"File exists? {(PROC / 'topic_top_terms_agg.csv').exists()}")

In [ ]:
df_terms_agg = pd.read_csv(PROC / "topic_top_terms_agg.csv")
print("--- Aggregated Model: Top 15 Terms ---")
display(df_terms_agg.head(15))

In [ ]:
df_terms_bal = pd.read_csv(PROC / "topic_top_terms_balanced.csv")
print("\n--- Balanced Model: Top 15 Terms ---")
display(df_terms_bal.head(15))

In [ ]:
df_sent_topic = pd.read_csv(PROC / "sentiment_by_topic.csv")
print("\n--- Sentiment Distribution per Topic (Aggregated) ---")
display(df_sent_topic)

In [ ]:
df_sent_plat = pd.read_csv(PROC / "sentiment_by_platform.csv")
print("\n--- Sentiment Distribution per Platform ---")
display(df_sent_plat)

In [ ]:
df_unigrams = pd.read_csv(PROC / "unigram_freq.csv")
print("\n--- Most Frequent Unigrams (Overall, Stopwords Removed) ---")
display(df_unigrams.head(15))

In [ ]:
df_strengths = pd.read_csv(PROC / "topic_strengths.csv")
print("\n--- Topic Strengths (Balanced Model) ---")
display(df_strengths.head(10))

In [ ]:
print("\n--- Visualizing Sentiment vs. Engagement ---")

df_sentiment = pd.read_csv(PROC / "sentiment_scored_full.csv")
df_corpus = pd.read_csv(RAW / "cleaned_corpus.csv")
df_sentiment['like_count'] = df_corpus['like_count']
df_sentiment['text_length'] = df_sentiment['cleaned_text'].str.len().fillna(0)

# --- Plotting ---
plt.figure(figsize=(10, 6))

# plot like_count vs. comment length, colored by negativity
plt.scatter(
    df_sentiment['text_length'],
    df_sentiment['like_count'],
    alpha=0.2, # transparency
    c=df_sentiment['neg'], # color by the 'neg' score
    cmap='Reds'            
)

plt.title('Like Count vs. Comment Length (Colored by Negativity)')
plt.xlabel('Comment Length (Characters)')
plt.ylabel('Like Count')
plt.xscale('log') # for length
plt.yscale('log') # for likes (some are viral)
plt.colorbar(label='Negative Sentiment Score')
plt.savefig(FIG / 'likes_vs_length_by_sentiment.png', dpi=160, bbox_inches='tight')
plt.close()

In [ ]:
import networkx as nx

print("\n--- Visualizing Social Network of Top Bigrams ---")

# load bigram data
df_bigram = pd.read_csv(PROC / "bigram_freq.csv")

# filter to only the strongest connections (e.g., top 75 bigrams)
df_top_bigrams = df_bigram.head(75).copy()

# split the bigram 'term' into two columns
df_top_bigrams[['word1', 'word2']] = df_top_bigrams['term'].str.split(' ', expand=True)

# create a graph from the pandas edgelist
G = nx.from_pandas_edgelist(
    df_top_bigrams,
    source='word1',
    target='word2',
    edge_attr='freq'
)

# --- Plot the graph ---
plt.figure(figsize=(14, 14))

pos = nx.spring_layout(G, k=0.7, iterations=50, seed=42)

# get edge weights for line thickness
weights = [G[u][v]['freq'] for u,v in G.edges()]
norm_weights = [(w / max(weights)) * 8 + 1 for w in weights] # Normalize

# get node sizes based on how connected they are
degrees = dict(G.degree())
node_sizes = [v * 300 for v in degrees.values()]

nx.draw(
    G,
    pos,
    with_labels=True,
    node_color='skyblue',
    node_size=node_sizes,
    font_size=10,
    width=norm_weights,
    edge_color='gray',
    alpha=0.8
)

plt.title('Social Network Analysis of Top 75 Bigrams')
plt.savefig(FIG / 'sna_top_75_bigrams.png', dpi=160, bbox_inches='tight')
plt.close()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

print("\n--- Visualizing Top 10 Comments by Engagement ---")

top_10_df = (
    df_sentiment[df_sentiment['source'] == 'youtube']
    .nlargest(10, 'like_count')
    .sort_values('like_count', ascending=False)
)

top_10_df['comment_label'] = top_10_df['cleaned_text'].str.slice(0, 75) + '...'
top_10_df['comment_label'] = top_10_df.apply(
    lambda row: f"Comment #{row.name}\n(Likes: {int(row['like_count'])})", axis=1
)

# Plot the horizontal bar chart
plt.figure(figsize=(12, 8))
ax = sns.barplot(
    data=top_10_df,
    x='like_count',
    y='comment_label',
    hue='sent_label',
    palette={'pos': 'green', 'neu': 'gray', 'neg': 'red'},
    dodge=False,
    hue_order=['neg', 'neu', 'pos']
)

ax.set_title('Top 10 YouTube Comments by Like Count', fontsize=16)
ax.set_xlabel('Like Count')
ax.set_ylabel('Comment')
plt.legend(title='Sentiment')
plt.tight_layout()
plt.savefig(FIG / 'top_10_comments_by_likes.png', dpi=160)
plt.show()

print("Saved: top_10_comments_by_likes.png")

# display the text of the top 3 comments
print("\n--- Text of Top 3 Comments ---")
for idx, row in top_10_df.head(3).iterrows():
    print(f"Likes: {int(row['like_count'])} | Sentiment: {row['sent_label'].upper()}")
    print(f"Comment: {row['cleaned_text']}\n")